# 📗 추론 가속: 서버 설정을 vllm bench sweep serve로 비교하고 정하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<div style="background:#ffffff;color:#202124;font-size:17px;line-height:1.6;padding:6px 10px;border-left:4px solid #54769b;">
<strong style="color:#1f3a5f;font-size:19px;">실습 환경</strong><br>
<strong style="font-size:1.13em;color:#1f3a5f;">실행 위치:</strong> VS Code의 Remote - SSH로 RunPod에 접속합니다. 노트북과 터미널, vLLM 0.27.0 서버와 벤치마크는 모두 같은 파드에서 실행합니다. <strong style="font-size:1.13em;color:#1f3a5f;">내 PC에서 실행하는 코드는 없습니다.</strong><br>
<strong style="font-size:1.13em;color:#1f3a5f;">준비:</strong> 딥러닝 학습 원리 단원 <a style="color:#245fa0;text-decoration:underline;" href="../day65_딥러닝_학습원리/부록_RunPod_GPU에서_실습하기.ipynb">RunPod 부록 5~7절</a>의 원격 접속 방식대로 저장소 폴더를 엽니다. 노트북 커널은 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.venv</code>를 고릅니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">자료:</strong> 교안 01에서 생성한 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code> 인증 키와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/</code> 입력 파일을 그대로 씁니다. 결과도 같은 폴더에서 바로 분석합니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">서버:</strong> 교안 01에서 띄운 상담 서버를 <strong style="font-size:1.13em;color:#1f3a5f;">그대로 이어 씁니다.</strong> 새로 띄우지 않아도 되며, 서버가 꺼져 있을 때만 1-2에서 다시 띄웁니다. 2절(접두어 캐싱)까지 이 서버를 쓰고 2-3에서 끕니다. 3절의 sweep은 비교할 서버(기준, MTP)를 직접 띄우고 끕니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">최소 권장:</strong> GPU VRAM 24GB 1장 · RAM 16GB · 디스크 80GB. 서버는 하나만 실행합니다.
</div>
</div>

#### **추론 가속 서버 설정**

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">모델과 GPU는 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">서버 옵션만 바꿔, 같은 GPU에서 지연을 줄이거나 출력 처리량을 늘리는 설정</strong>입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> 서버를 늘리기 전에, 같은 GPU로 지연 목표를 지키며 처리하는 요청을 늘립니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 참고 문서가 긴 RAG 서버와 상담 서버, 동시 요청이 몰리는 사내 챗봇 서버의 설정 결정입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/16_acceleration_concept.png" width="1050" alt="같은 GPU 서버에서 접두어 캐싱, KV 캐시 자료형, 동시 실행 상한, 투기적 디코딩 설정을 바꾸고, 지연과 처리량을 비교해 설정을 고릅니다.">

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01의 노무 상담 서버로 추론 가속 설정 네 가지를 배웁니다. 상담 서버에서 접두어 캐싱의 캐시 적중을 확인한 뒤, 서버 설정마다 같은 벤치마크를 반복하는 sweep(스윕)을 RunPod에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code> 명령 하나로 실행</strong>해 기준 설정과 MTP 설정(모델에 들어 있는 다중 토큰 예측 층으로 다음 토큰들을 미리 추측해 생성을 빠르게 하는 설정)을 비교하고, 예상 동시 요청 수에서 지연 목표를 지키는 설정을 고릅니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/10_lesson02_goals.png" width="1050" alt="학습 목표 그림: 1 접두어 캐싱, 2 KV 캐시 자료형 fp8, 3 동시 실행 상한, 4 투기적 디코딩">

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

**학습 목표** (그림의 번호와 같습니다)

1. 앞부분이 같은 요청에서 접두어 캐싱의 캐시 적중을 확인하고, 캐싱을 끈 서버와 켠 서버의 TTFT와 출력 처리량을 측정 예로 비교합니다.
2. `--kv-cache-dtype fp8`이 KV 캐시에 담는 토큰 수를 늘리는 원리와 쓰는 때를 설명합니다.
3. `--max-num-seqs`가 동시 실행 요청 수를 제한하는 방식과 TPOT·TTFT·출력 처리량의 절충 관계를 설명합니다.
4. MTP(다중 토큰 예측 층)로 초안을 만드는 투기적 디코딩의 수용 길이와 동시 요청 수별 효과를 기준 설정과 `vllm bench sweep serve`로 한 번에 측정해 비교하고, 서버 설정을 정합니다.

</div>

## 1. 실습 준비

### 1-1. RunPod 원격 노트북 준비

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01의 본실습과 함께 따라하기를 마친 <strong style="font-size:1.13em;color:#1f3a5f;">VS Code 원격 창</strong>에서 이 노트북을 열고 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.venv</code> 커널을 고릅니다. 아래 셀은 교안 01과 같은 준비 코드(경로, 설정값, 데이터, 토크나이저)와 LangChain 채팅 모델 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">llm</code>을 모은 것입니다.</p>
<p style="margin:6px 0;">인증 키는 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 읽고, 서버 주소는 같은 파드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:8000</code>을 씁니다. 데이터는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/본실습/labor_reference.json</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/본실습/labor_questions.jsonl</code>이고, 벤치마크 입력은 교안 01이 생성한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/labor_prompts.jsonl</code>입니다.</p>
<p style="margin:6px 0;">파드를 새로 만들어 키나 입력 파일이 없다면 교안 01의 준비와 입력 파일 생성부터 마칩니다.</p>
</div>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
# ---- 라이브러리, 경로, 설정값 ----
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from transformers import AutoTokenizer

material_dir = Path(".").resolve()  # 이 단원 폴더(정답 노트북은 "..")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
bench_dir = output_dir / "bench"           # 같은 파드에서 벤치마크가 저장한 결과 JSON
sweep_dir = output_dir / "sweep"           # 같은 파드에서 sweep이 저장한 summary.csv
bench_dir.mkdir(parents=True, exist_ok=True)
sweep_dir.mkdir(parents=True, exist_ok=True)
# 터미널에서 생성해 원격 저장소의 .env에 저장한 키를 읽습니다
load_dotenv(material_dir.parent / ".env", override=True)
SERVER_URL = "http://127.0.0.1:8000"  # 노트북과 서버가 같은 RunPod에 있습니다
VLLM_API_KEY = os.environ["VLLM_API_KEY"]

# 설정값
MODEL_NAME = "Qwen/Qwen3.5-4B"            # 서버가 불러온 허브 모델(토크나이저도 이 이름으로 받습니다)
SERVED_NAME = "qwen3.5-4b"               # 서버 이름(--served-model-name). 요청의 model에 넣습니다
MAX_MODEL_LEN = 8192                  # 참고 조문이 든 시스템 메시지 + 질문 + 답을 담는 최대 길이
MAX_TOKENS = 256                     # 요청마다 생성 상한(지난 단원과 같은 값)
TTFT_TARGET_MS = 1000                # 지연 목표: 첫 토큰까지 1000ms 안
TPOT_TARGET_MS = 50                  # 지연 목표: 출력 토큰 하나에 50ms 안

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
import platform

import matplotlib.pyplot as plt
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스: 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

# 지난 단원과 같은 지침 문장. 시스템 메시지는 이 문장 뒤에 참고 조문을 붙여 만듭니다
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."

# ---- JSONL 읽기(교안 01과 같은 함수) ----
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# ---- 데이터, 토크나이저 ----
# 노무 상담의 참고 조문과 질문, 시스템 메시지 = 지침 문장 + 빈 줄 + [참고 조문] + 조문들
with open(data_dir / "본실습" / "labor_reference.json", encoding="utf-8") as f:
    labor_reference = json.load(f)
labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")
labor_articles = [article["text"] for article in labor_reference["articles"]]
system_prompt = SYSTEM_PROMPT + "\n\n[참고 조문]\n" + "\n".join(labor_articles)

# 서버와 같은 모델의 토크나이저를 씁니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ---- 상담 서버에 요청할 채팅 모델(교안 01과 같습니다) ----
from langchain_openai import ChatOpenAI

# 같은 파드의 서버 주소와 인증 키를 사용하는 채팅 모델(지난 단원 ChatOpenAI와 같은 모양)
llm = ChatOpenAI(
    model=SERVED_NAME,                 # 서버 이름(--served-model-name)
    base_url=f"{SERVER_URL}/v1",       # 같은 파드의 로컬 서버 주소 + /v1
    api_key=VLLM_API_KEY,              # 서버를 띄울 때 정한 --api-key
    temperature=0,
    max_tokens=MAX_TOKENS,
    stream_usage=True,                 # base_url을 직접 정하면 기본값이 꺼짐이라, 스트리밍 응답에서도 토큰 사용량을 받으려고 켭니다
)

# 같은 파드에서 교안 01이 생성한 파일(조문이 앞, 질문이 뒤)
labor_prompts_path = output_dir / "labor_prompts.jsonl"
print("질문 수:", len(labor_questions), "| 시스템 메시지 토큰 수:", len(tokenizer.encode(system_prompt)), "| 입력 파일:", labor_prompts_path.name)

### 1-2. 상담 서버 확인

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 01의 상담 서버가 떠 있으면 그대로 씁니다.</strong> RunPod 터미널 1에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code> 뒤로 오류 없이 로그가 이어지고 있으면 떠 있는 것입니다. 서버를 이어 쓰면 2절을 마칠 때까지 서버를 새로 띄우지 않습니다.</p>
<p style="margin:6px 0;">상담 서버만 껐다면 원격 VS Code에서 <strong style="font-size:1.13em;color:#1f3a5f;">Terminal → New Terminal</strong>로 터미널 1을 엽니다. 새 터미널은 원격 저장소 폴더에서 시작합니다. 아래 준비 블록이 단원 폴더로 이동해 원격 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 키를 불러오면, 다음 블록으로 서버를 다시 띄웁니다. 교안 01에서 생성한 키를 그대로 씁니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 1**

```bash
# VS Code에서 연 원격 저장소(실습자료) 폴더의 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다
mkdir -p output/bench output/sweep
# 같은 원격 저장소의 .env에서 이미 생성한 키를 불러옵니다
VLLM_API_KEY="$(uv run python -c 'from dotenv import dotenv_values; print(dotenv_values("../.env")["VLLM_API_KEY"])')" &&
export VLLM_API_KEY OPENAI_API_KEY="$VLLM_API_KEY"
```

**복사해서 실행 · RunPod 터미널 1**

```bash
uv run vllm serve Qwen/Qwen3.5-4B \
  --served-model-name qwen3.5-4b \
  --max-model-len 8192 \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-prompt-tokens-details \
  --enable-prefix-caching \
  --host 127.0.0.1 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b_server.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. 서버 출력은 RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/qwen3.5-4b_server.log</code>에도 남습니다. <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">같은 VS Code 원격 창의 노트북</strong>으로 돌아옵니다.</p>
<p style="margin:6px 0;">첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
</div>

## 2. 접두어 캐싱

### 2-1. 블록 단위로 다시 쓰는 KV 캐시

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">접두어 캐싱(Automatic Prefix Caching)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">요청마다 같은 앞부분의 KV 캐시를 블록 단위로 남겨 두고, 다음 요청의 앞부분이 같으면 그 블록의 prefill을 건너뛰고 다시 쓰는 기능입니다. 서버 옵션은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>입니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시:</strong> 교안 01에서 본, 앞 토큰들의 K·V 계산 결과를 GPU 메모리에 저장해 둔 것입니다. 다음 토큰을 만들 때 앞 토큰을 다시 계산하지 않고 이 값을 읽습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">블록:</strong> KV 캐시를 정해진 토큰 수씩 나눠 담는 칸입니다(지난 단원 교안 01 5-3). 다 찬 블록을 다시 씁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">블록을 알아보는 방법:</strong> 앞 블록의 식별값과 현재 블록의 토큰으로 짧은 식별값(해시)을 만들어, 맨 앞부터 같은 토큰이 이어지는 블록을 찾습니다.</li>
</ul>
</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 시스템 메시지, 참고 문서, 이전 대화처럼 요청마다 같은 앞부분(공통 접두어, common prefix)의 prefill을 줄여 TTFT를 줄이고 GPU 시간을 아낍니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">누구의 요청이든 함께 씁니다:</strong> 서버 하나의 접두어 캐시는 모든 요청이 공유합니다. 사용자가 달라도 앞부분 토큰이 같으면 다시 씁니다. 예: A 사용자의 질문이 남긴 시스템 메시지 블록을, 같은 시스템 메시지로 시작하는 B 사용자의 질문이 그대로 씁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">켜는 법:</strong> vLLM 0.27.0은 Qwen3.5처럼 attention 층과 Gated DeltaNet 층(토큰마다 KV를 쌓지 않고 크기가 정해진 상태 하나만 저장하는 층)을 섞어 쓰는 하이브리드 모델에서 이 기능을 기본으로 끕니다. 그래서 교안 01의 서버 명령이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>으로 켜고, 이 옵션이 없으면 이 모델은 캐싱 없이 돕니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">캐시 적중 토큰(cached tokens):</strong> 입력 토큰 가운데 캐시에서 다시 쓴 토큰 수입니다. 서버 명령의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prompt-tokens-details</code>가 이 값을 응답에 넣고, LangChain 응답에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage_metadata[&quot;input_token_details&quot;][&quot;cache_read&quot;]</code>로 읽습니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/11_prefix_caching.png" width="1050" alt="시스템 메시지가 같은 둘째 요청은 꽉 찬 블록만 다시 쓰고, 덜 찬 마지막 블록의 시스템 메시지 끝부분과 질문은 새로 계산합니다. 질문을 앞에 두면 첫 블록부터 달라 다시 쓸 블록이 없습니다.">

<p style="margin:6px 0;">블록은 통째로 같아야 다시 씁니다. 블록 하나가 528토큰이고 입력이 교안 01에서 센 약 4,980토큰(시스템 메시지 약 4,930토큰)인 경우를 따라가 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">시스템 메시지 가운데 앞 블록 9개(528 × 9 = 4,752토큰)가 꽉 찹니다.</li>
<li style="margin:3px 0;">같은 시스템 메시지의 둘째 요청은 이 블록 9개를 다시 쓰고, 블록을 채우지 못한 나머지 시스템 메시지 토큰과 질문만 prefill 합니다.</li>
<li style="margin:3px 0;">질문이 맨 앞에 오면 첫 블록부터 요청마다 내용이 달라 다시 쓸 블록이 없습니다.</li>
</ol>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">블록 크기가 큰 이유:</strong> Qwen3.5는 토큰마다 KV를 쌓는 attention 층과 크기가 정해진 상태 하나를 저장하는 Gated DeltaNet 층을 섞어 씁니다. vLLM은 attention 블록 하나가 이 상태 하나보다 작지 않도록 블록을 크게 잡습니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인할 로그:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Setting attention block size to ... tokens</code>가 그 크기입니다. 캐시 적중은 이 블록 크기 단위로 일어납니다.</li>
</ul>
</li>
</ul>
</div>

### 2-2. 캐시 적중 확인

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">상담 서버에 메시지 세 개를 차례로 보냅니다. 처음 둘은 같은 시스템 메시지(조문이 앞)에 서로 다른 질문이고, 셋째는 둘째 질문을 질문이 앞인 메시지(시스템 메시지는 지침 문장만, user 메시지는 질문 뒤에 조문)로 바꾼 것입니다.</p>
</div>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
# 보낼 메시지 세 개: (프롬프트 순서, 질문, 메시지)
cache_requests = [
    ("조문을 앞에", labor_questions[0]["question"], [("system", system_prompt), ("human", labor_questions[0]["question"])]),
    ("조문을 앞에", labor_questions[1]["question"], [("system", system_prompt), ("human", labor_questions[1]["question"])]),
    ("질문을 앞에", labor_questions[1]["question"], [("system", SYSTEM_PROMPT), ("human", labor_questions[1]["question"] + "\n\n[참고 조문]\n" + "\n".join(labor_articles))]),
]

# 차례로 보내고(스트리밍 없이 temperature=0으로), 입력 토큰 수와 그 가운데 캐시에서 다시 쓴 토큰 수를 모읍니다
cache_hit_rows = []
for order, question, messages in cache_requests:
    response = llm.invoke(messages)
    cache_hit_rows.append({
        "프롬프트 순서": order,
        "질문": question,
        "입력 토큰 수(input_tokens)": response.usage_metadata["input_tokens"],
        "캐시 적중 토큰(cache_read)": response.usage_metadata["input_token_details"]["cache_read"],
    })

cache_hit_table = pd.DataFrame(cache_hit_rows)
display(cache_hit_table)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 1은 서버를 실행 중이므로 RunPod 터미널 2에서 블록 크기를 확인합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 2**

```bash
# 새 원격 터미널이면 이 블록부터 실행합니다
# VS Code에서 연 원격 저장소(실습자료) 폴더의 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다
mkdir -p output/bench output/sweep
# 같은 원격 저장소의 .env에서 이미 생성한 키를 불러옵니다
VLLM_API_KEY="$(uv run python -c 'from dotenv import dotenv_values; print(dotenv_values("../.env")["VLLM_API_KEY"])')" &&
export VLLM_API_KEY OPENAI_API_KEY="$VLLM_API_KEY"
```

**RunPod 터미널 2 · 화면에서 확인**

```bash
grep "attention block size" output/qwen3.5-4b_server.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">둘째 행(조문을 앞에):</strong> 시스템 메시지가 같아 앞 블록들을 다시 씁니다. 캐시 적중 토큰은 화면의 블록 크기의 정수배이고, 나머지 시스템 메시지 끝부분과 질문만 새로 prefill 합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">셋째 행(질문을 앞에):</strong> 같은 질문과 같은 조문이지만 첫 블록부터 앞의 요청과 내용이 달라 캐시 적중 토큰이 0입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">첫 행:</strong> 같은 시스템 메시지를 이미 보낸 서버를 이어 쓰면 적중할 수 있고, 서버를 새로 띄웠다면 다시 쓸 블록이 없어 0입니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">다른 창구의 요청을 많이 처리해 노무 블록이 지워졌다면, 이어 쓴 서버에서도 0일 수 있습니다.</li>
<li style="margin:3px 0;">이 셀을 다시 실행하면 셋째 행도 앞 실행의 블록을 다시 써 적중할 수 있습니다.</li>
</ul>
</li>
</ul>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 예: 접두어 캐싱을 끈 서버와 켠 서버</strong></p>
<p style="margin:6px 0;">RunPod L4 24GB, vLLM 0.27.0에서 교안 01과 같은 조건(노무 입력, 동시 요청 16, 요청 64개)으로 접두어 캐싱만 끄고 켜서 측정한 예입니다. 직접 다시 측정하지 않으며, GPU가 다르면 숫자도 달라집니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">항목</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">접두어 캐싱 끔</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">접두어 캐싱 켬</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">걸린 시간(초)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">89.6</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">36.2</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">출력 처리량(토큰/초)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">107.4</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">274.1</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">평균 TTFT(ms)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">2,892</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">280</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">p95 TTFT(ms)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">11,214</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">760</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">평균 TPOT(ms)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">124.1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">47.8</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TTFT:</strong> 캐싱을 켜면 요청마다 시스템 메시지 앞부분 4,752토큰(528토큰 블록 9개)을 다시 써, 평균 TTFT가 2,892ms에서 280ms로 줄었습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TPOT:</strong> 캐싱을 끄면 새 요청마다 약 5,000토큰의 prefill이 생성 단계에 섞여, 이미 답을 만들던 요청도 느려집니다. 그래서 평균 TPOT도 124.1ms에서 47.8ms로 줄었습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">비용:</strong> 같은 GPU의 출력 처리량이 107.4에서 274.1토큰/초로 늘어, 출력 토큰당 GPU 비용은 약 39%(107.4 / 274.1)로 줄었습니다.</li>
</ul>
</div>

### 2-3. 상담 서버 끄기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">상담 서버로 하는 실습은 여기까지입니다. 3절의 sweep이 같은 GPU와 포트 8000에 서버를 설정마다 차례로 띄우므로 지금 상담 서버를 끕니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code> 서버를 띄운 터미널)에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>를 누릅니다. 명령을 입력할 수 있게 되면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>를 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU 메모리 사용량(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Memory-Usage</code>)이 서버를 띄우기 전 수준으로 줄었으면</strong> 서버가 꺼진 것입니다. 모델 가중치와 KV 캐시가 쓰던 메모리가 이때 반환됩니다. RunPod에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>의 프로세스 목록이 비어 보일 수 있어 메모리 사용량으로 판단합니다.</p>
</div>

## 3. vllm bench sweep serve로 측정 먼저 실행하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">sweep은 서버 시작 두 번과 측정 여섯 번이 이어져 몇 분 넘게 걸립니다. 그래서 <strong style="font-size:1.13em;color:#1f3a5f;">측정을 먼저 실행해 두고, 기다리는 동안 4~6절에서 각 설정의 원리를 배운 뒤 7절에서 결과를 읽습니다.</strong></p>
</div>

### 3-1. vllm bench sweep serve가 하는 일

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">sweep(스윕)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">설정 값을 하나씩 바꿔 가며 같은 측정을 차례로 반복하는 일입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code>는 서버 설정마다 서버를 띄우고 같은 벤치마크를 돌리는 일을 명령 하나로 해 줍니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정용 서버:</strong> sweep이 설정마다 파드 안(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1</code>)에서만 띄우는 서버입니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code>의 순서</strong></p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">서버 설정 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--serve-params</code>)의 설정마다 서버 명령(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--serve-cmd</code>) 뒤에 그 설정의 옵션을 더해 측정용 서버를 띄우고, 준비될 때까지 기다립니다.</li>
<li style="margin:3px 0;">벤치마크 조건 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-params</code>)의 조건마다 벤치마크 명령(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-cmd</code>)에 조건을 더해 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code>를 실행하고 결과 JSON을 저장합니다. 측정이 끝날 때마다 서버의 접두어 캐시를 비워, 앞 측정이 다음 측정에 남지 않게 합니다.</li>
<li style="margin:3px 0;">서버를 끄고 다음 설정으로 넘어갑니다.</li>
<li style="margin:3px 0;">모든 결과를 실험 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--output-dir</code> 아래 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--experiment-name</code> 폴더)의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code> 한 파일에 모읍니다. 측정 하나가 한 행입니다.</li>
</ol>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정 파일을 옵션으로 바꾸는 규칙</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">키는 옵션 이름의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_</code>로 쓴 것입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;kv_cache_dtype&quot;: &quot;fp8&quot;}</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--kv-cache-dtype fp8</code>이 됩니다.</li>
<li style="margin:3px 0;">값이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">false</code>이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--no-</code> 옵션이 됩니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;enable_prefix_caching&quot;: false}</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--no-enable-prefix-caching</code>입니다.</li>
<li style="margin:3px 0;">값이 딕셔너리이면 JSON 문자열로 넣습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--speculative-config</code>에 쓰는 형식입니다.</li>
<li style="margin:3px 0;">명령에 이미 있는 옵션이면 그 값을 바꿉니다. 벤치마크 조건의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dataset_path</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-cmd</code>의 입력 파일을 바꿉니다.</li>
<li style="margin:3px 0;">파일을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;이름&quot;: {설정}}</code> 모양으로 쓰면 그 이름이 결과 폴더와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_benchmark_name</code> 열에 들어갑니다. 두 파일에 모두 이름을 붙이면 이 열에 벤치마크 조건 이름만 남으므로, 이 교안은 서버 설정 파일에만 이름을 붙이고 벤치마크 조건 파일은 이름 없는 리스트로 씁니다.</li>
</ul>
</div>

### 3-2. 비교할 서버 설정과 벤치마크 조건

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버 설정은 기준과 MTP 둘입니다. MTP는 모델 안의 다중 토큰 예측 층으로 초안을 만드는 투기적 디코딩 설정이며, 원리는 기다리는 동안 6절에서 배웁니다. 설정 하나마다 서버를 한 번 더 띄워야 하므로 4, 5절의 설정은 측정하지 않습니다. 접두어 캐싱은 두 서버 모두 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>으로 켜 둡니다.</p>
<p style="margin:6px 0;">벤치마크 조건은 셋입니다. 시간을 줄이려고 8절의 비교와 9절 함께 따라하기에 쓰는 조건만 둡니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">노무, 동시 요청 1(요청 8개):</strong> 낮은 부하입니다. 요청 하나를 빠르게 하는 MTP의 효과를 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">노무, 동시 요청 64(요청 64개):</strong> 높은 부하입니다. GPU 계산이 요청들로 차 있을 때 MTP의 효과가 어떻게 바뀌는지 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">전월세, 동시 요청 16(요청 16개):</strong> 함께 따라하기에서 씁니다. 입력 파일은 교안 01 함께 따라하기에서 생성한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/lease_prompts.jsonl</code>입니다.</li>
</ul>
<p style="margin:6px 0;">설정 2개 x 조건 3개로 측정은 6번입니다.</p>
<p style="margin:6px 0;">두 파일은 같은 파드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/serve_params.json</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench_params.json</code>에 저장합니다. 이어서 터미널의 sweep이 이 파일을 바로 읽습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dataset_path</code>는 단원 폴더 기준 경로입니다.</p>
</div>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
# 비교할 서버 설정: 이름별로 서버 명령 뒤에 더할 옵션
serve_params = {
    "기준": {},                                                                        # 옵션을 바꾸지 않은 서버
    "MTP": {"speculative_config": {"method": "mtp", "num_speculative_tokens": 2}},     # --speculative-config '{...}'
}

# 서버마다 보낼 벤치마크 조건: 노무는 낮은 부하(동시 요청 1)와 높은 부하(동시 요청 64), 전월세는 동시 요청 16
bench_params = [
    {"dataset_path": "output/labor_prompts.jsonl", "max_concurrency": 1, "num_prompts": 8},
    {"dataset_path": "output/labor_prompts.jsonl", "max_concurrency": 64, "num_prompts": 64},
    {"dataset_path": "output/lease_prompts.jsonl", "max_concurrency": 16, "num_prompts": 16},
]

# sweep이 읽는 JSON 파일로 저장합니다
serve_params_path = output_dir / "serve_params.json"
bench_params_path = output_dir / "bench_params.json"
serve_params_path.write_text(json.dumps(serve_params, ensure_ascii=False, indent=2), encoding="utf-8")
bench_params_path.write_text(json.dumps(bench_params, indent=2), encoding="utf-8")
print("서버 설정:", list(serve_params), "| 벤치마크 조건:", len(bench_params), "개")

### 3-3. RunPod에서 측정 실행

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">2-3에서 상담 서버를 껐는지 확인합니다. 서버가 아직 실행 중이면 RunPod 터미널 1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끄고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>로 GPU 메모리가 반환됐는지 확인한 뒤 아래 명령을 실행합니다. sweep이 포트 8000에 측정용 서버를 띄우므로 상담 서버가 떠 있으면 겹칩니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 1**

```bash
uv run vllm bench sweep serve \
  --serve-cmd 'vllm serve Qwen/Qwen3.5-4B --served-model-name qwen3.5-4b --max-model-len 8192 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs "{\"enable_thinking\": false}" --enable-prompt-tokens-details --enable-prefix-caching --host 127.0.0.1 --port 8000' \
  --bench-cmd 'vllm bench serve --backend openai --base-url http://127.0.0.1:8000 --model qwen3.5-4b --tokenizer Qwen/Qwen3.5-4B --dataset-name custom --dataset-path output/labor_prompts.jsonl --skip-chat-template --custom-output-len 256 --temperature 0 --seed 0 --metric-percentiles 50,95,99 --goodput ttft:1000 tpot:50' \
  --serve-params output/serve_params.json \
  --bench-params output/bench_params.json \
  --num-runs 1 \
  --server-ready-timeout 600 \
  --output-dir output/sweep \
  --experiment-name labor-settings \
  --show-stdout 2>&1 | tee output/sweep/labor-settings.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--serve-cmd '...'</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">설정마다 띄울 서버 명령입니다. 상담 서버 명령에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>를 뺀 모양이며, 같은 파드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1:8000</code>에서 요청을 받습니다. 키는 터미널의 환경변수를 물려받습니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-cmd '...'</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">측정마다 실행할 벤치마크 명령입니다. 교안 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code>에서 동시 요청 수, 요청 수, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--percentile-metrics</code>, 결과 저장 옵션을 뺀 모양입니다. 동시 요청 수와 요청 수는 벤치마크 조건 파일이 붙이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--percentile-metrics</code>와 결과 저장은 sweep이 붙입니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--serve-params</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-params</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">앞 셀에서 저장한 서버 설정 파일과 벤치마크 조건 파일입니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--num-runs 1</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">조건마다 측정 횟수입니다(기본 3). 시간을 줄이려고 한 번만 측정합니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--server-ready-timeout 600</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">서버 준비를 기다리는 시간(초, 기본 300)입니다. MTP 서버처럼 준비가 오래 걸리는 설정이 있어 늘립니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--output-dir</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--experiment-name</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">결과가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/sweep/labor-settings/</code> 폴더에 쌓입니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--show-stdout</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">서버와 벤치마크의 출력을 화면에 보입니다. 주지 않으면 서버 출력이 버려져 시작 로그의 KV 캐시 줄을 확인할 수 없습니다</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">화면 출력은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/sweep/labor-settings.log</code>에도 남깁니다. 측정용 서버의 API 키는 터미널 준비 블록에서 정한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>를 물려받고, 벤치마크는 같은 값의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OPENAI_API_KEY</code>를 요청에 붙입니다. 터미널 1을 새로 열었다면 1-2의 터미널 1 준비 블록부터 실행합니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">sweep은 서버 시작 두 번과 측정 여섯 번이 이어져 몇 분 넘게 걸립니다. 화면에 서버 시작 로그가 올라오고, 측정마다 벤치마크 결과 블록이 찍히면 정상입니다. <strong style="font-size:1.13em;color:#1f3a5f;">터미널 1은 그대로 두고, 기다리는 동안 4절부터 읽습니다.</strong> 끝나면 7절에서 결과를 확인합니다. 서버 시작이 한 번에 몇 분씩 걸려, 서버 설정을 하나 더할 때마다 sweep이 그만큼 길어집니다.</p>
<p style="margin:6px 0;">sweep은 uv run 아래에서 실행되므로 안쪽 명령은 vllm으로 바로 시작합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">다시 실행할 때</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">처음부터 다시:</strong> 같은 실험 이름 폴더가 있으면 sweep이 시작하자마자 멈춥니다. 앞 실험 폴더는 지우지 않고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">mv output/sweep/labor-settings output/sweep/labor-settings-old</code>로 옮겨 둔 뒤 같은 명령을 다시 실행합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이어 하기:</strong> 중단된 실행은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--show-stdout</code> 앞에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--resume</code>을 붙이고, 마지막 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee -a</code>로 바꿔 로그를 이어 씁니다. 결과가 이미 있는 측정은 건너뜁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">중간에 멈췄을 때:</strong> sweep을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 멈추거나 오류로 끝나면 측정용 서버가 GPU를 잡은 채 남아 있을 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>의 GPU 메모리 사용량이 줄지 않았으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pgrep -af &quot;vllm serve&quot;</code>로 남은 vLLM 프로세스의 PID를 찾아 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">kill &lt;PID&gt;</code>로 끈 뒤 다시 실행합니다.</li>
</ul>
</div>

## 4. KV 캐시 자료형: --kv-cache-dtype

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">시작 로그의 KV 캐시 토큰 수가 클수록 긴 요청과 많은 동시 요청을 함께 담습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시 자료형(KV cache dtype, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--kv-cache-dtype</code>)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">KV 캐시에 값을 저장하는 자료형입니다. 기본은 모델 자료형(bf16, 값 하나 2바이트)이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--kv-cache-dtype fp8</code>은 fp8(8-bit floating point, 8비트 부동소수, 값 하나 1바이트)로 저장해 같은 메모리에 토큰을 더 담습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 긴 문맥이나 많은 동시 요청으로 KV 캐시가 모자랄 때 담을 수 있는 토큰 수를 늘립니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/14_kv_cache_dtype.png" width="1050" alt="fp8은 attention KV 값을 1바이트로 저장하지만 Gated DeltaNet 상태는 그대로입니다. 그림의 약 1.5배는 예시이며 실제로 담는 토큰 수의 증가율은 모델과 설정에 따라 다릅니다.">

<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">그림의 증가율:</strong> 약 1.5배는 설명용 예입니다. 실제로 담는 토큰 수의 증가율은 모델과 설정에 따라 달라집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">저장 공간:</strong> Qwen3.5의 KV 캐시 메모리에는 두 가지가 함께 들어갑니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">attention 층의 K·V:</strong> 토큰마다 쌓입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--kv-cache-dtype fp8</code>이 줄이는 것은 이 값뿐입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Gated DeltaNet 층의 상태:</strong> 토큰마다 쌓지 않고, 요청 하나에 지금까지 읽은 내용을 요약한 크기가 정해진 값을 하나 저장합니다. fp8 옵션의 대상이 아니라 크기가 그대로입니다.</li>
<li style="margin:3px 0;">그래서 메모리의 일부만 줄어, 담는 토큰 수가 2배보다 적게 늘어납니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--mamba-cache-dtype</code>:</strong> Gated DeltaNet 상태의 자료형을 정하는 옵션입니다. 이름의 mamba는 K·V를 쌓지 않고 상태 하나만 저장하는 대표적인 모델 구조(Mamba)에서 왔습니다. vLLM은 Gated DeltaNet처럼 같은 방식으로 저장하는 층을 이 이름으로 묶어 부릅니다.</li>
</ul>
</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">캐시 적중:</strong> attention 블록 크기도 바뀔 수 있어 접두어 캐싱의 적중 단위가 달라집니다. 저장 공간이 늘어도 TTFT가 반드시 짧아지는 것은 아닙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">답 품질:</strong> 저장하는 값의 정밀도가 낮아져 답이 조금 바뀔 수 있으므로, SFT 지도 파인튜닝 단원(day70) 교안 02의 판사 평가로 다시 확인합니다.</li>
</ul>
<p style="margin:6px 0;">이 교안은 fp8 KV를 설명만 하고 측정하지 않습니다. 동시 요청이 많아 KV 캐시가 모자랄 때 기준 설정과 같은 부하로 측정해 비교합니다.</p>
</div>

## 5. 동시 실행 상한: --max-num-seqs

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 실행 상한(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--max-num-seqs</code>)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">스케줄러가 한 생성 단계에 함께 계산하는 요청 수의 상한입니다. 상한을 넘은 요청은 서버의 대기열(queue)에서 자리가 날 때까지 기다립니다. 24GB GPU에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm serve</code>의 기본값은 256입니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">스케줄러:</strong> 생성 단계마다 prefill과 decode에 어떤 요청을 함께 넣을지 고르는 vLLM의 부분입니다. 지난 단원의 연속 배치가 여기서 일어납니다.</li>
</ul>
</li>
<li style="margin:3px 0;">벤치마크의 동시 요청 수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-concurrency</code>)는 보내는 쪽의 상한이고, 동시 실행 상한은 서버 쪽의 상한입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 함께 계산하는 요청이 많아 TPOT가 길어질 때, 한 단계의 계산량을 줄입니다. 대신 더 많은 요청이 대기하면 TTFT가 길어질 수 있습니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/12_max_num_seqs_exact.png" width="1050" alt="요청 64개를 함께 처리할 메모리가 충분한 서버의 예입니다. 상한 16이면 처리 중 16개와 대기 48개, 상한 64이면 처리 중 64개와 대기 0개입니다.">

<p style="margin:6px 0;">요청 64개를 함께 처리할 수 있는 서버에서 상한을 16으로 낮춘 경우를 따라가 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">16개만 생성 단계에 들어가고 48개는 대기열에서 기다립니다.</li>
<li style="margin:3px 0;">한 단계에 계산하는 요청이 줄어, 단계가 짧아지고 TPOT도 짧아질 수 있습니다.</li>
<li style="margin:3px 0;">뒤 요청은 앞 요청이 끝날 때까지 기다려 TTFT가 길어질 수 있습니다. 함께 만드는 토큰도 줄어 출력 처리량을 잃을 수 있습니다.</li>
</ol>
<p style="margin:6px 0;">실제 동시 요청 수가 상한보다 적으면 상한을 낮춰도 제한이 걸리지 않습니다. 효과는 부하와 메모리 여유에 따라 달라지므로, 지연과 처리량을 함께 비교합니다.</p>
<p style="margin:6px 0;">이 교안은 동시 실행 상한을 설명만 하고 측정하지 않습니다. 요청이 몰려도 TPOT를 짧게 지켜야 할 때 상한을 줄여 같은 부하로 측정해 봅니다.</p>
</div>

## 6. 투기적 디코딩: --speculative-config

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청이 적을 때 GPU 계산 능력이 남는 이유</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">가중치 읽기:</strong> 가중치는 GPU 메모리(VRAM)에 올라가 있지만, 실제 계산을 하는 칩 안의 계산 장치는 저장 공간이 작아 가중치 전체를 담아 두지 못합니다. 그래서 decode는 생성 단계마다 모든 층의 가중치를 VRAM에서 계산 장치로 다시 읽어 옵니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">읽는 시간은 요청 수와 거의 상관없습니다:</strong> 한 단계에 한 번 읽은 가중치로 그 단계의 요청들을 함께 계산합니다. 계산 시간만 요청 수만큼 늘어납니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">예:</strong> 4B 모델(bf16, 약 8GB)을 L4 GPU(메모리 읽기 속도 초당 약 300GB)에서 돌리면 한 단계에 가중치를 읽는 데만 약 27ms가 걸리지만, 요청 하나의 토큰 계산은 1ms도 걸리지 않습니다. 요청이 하나면 대부분의 시간을 가중치가 도착하기를 기다리며 보내고, 계산 장치는 남습니다(8-1 측정 예에서 동시 요청 1의 기준 서버 TPOT 35.6ms).</li>
<li style="margin:3px 0;">투기적 디코딩은 이 남는 계산 능력을 씁니다. 한 번 읽은 가중치로 초안 토큰 여러 개를 함께 검증해, 한 단계에 토큰을 더 얻습니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">투기적 디코딩(speculative decoding)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">다음 토큰 몇 개를 작고 빠른 부품이 먼저 <strong style="font-size:1.13em;color:#1f3a5f;">미리 예측</strong>하고, 목표 모델(target model, 서비스하는 모델)이 한 단계에 그 예측을 <strong style="font-size:1.13em;color:#1f3a5f;">확인</strong>하는 생성 방법입니다. 미리 예측한 토큰을 <strong style="font-size:1.13em;color:#1f3a5f;">초안 토큰(draft token)</strong>이라고 합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 가중치 읽기 때문에 GPU 계산 능력이 남는 낮은 부하에서, 요청 하나의 생성 속도를 높입니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">1단계 미리 예측: MTP 층이 초안을 만듭니다</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">MTP(Multi-Token Prediction, 다중 토큰 예측):</strong> 모델을 학습할 때 다음 토큰 하나만이 아니라 <strong style="font-size:1.13em;color:#1f3a5f;">그 뒤에 올 토큰까지 함께 예측하도록 붙여 둔 작은 층(MTP 층)</strong>입니다. 이 교안의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Qwen/Qwen3.5-4B</code> 허브 모델에 들어 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">예측하는 방법:</strong> 목표 모델이 토큰을 하나 만들 때 계산한 결과(은닉 상태)를 MTP 층이 넘겨받아 그다음 토큰을 예측합니다. 초안이 2개면 앞에서 예측한 초안을 이어받아 하나를 더 예측합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">초안이 싼 이유:</strong> 목표 모델의 많은 층 대신 작은 층 하나만 계산하므로 금방 끝납니다. 초안용 작은 모델을 따로 내려받거나 띄우지 않아도 됩니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">2단계 확인: 목표 모델이 한 단계에 확인합니다</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">목표 모델을 한 번 계산(순전파)해 초안들을 함께 확인합니다. 앞에서부터 목표 모델이 고른 토큰과 같은 초안은 수용하고(답에 그대로 씁니다), 처음 틀린 자리에는 목표 모델이 고른 토큰 하나를 씁니다. 초안이 모두 맞으면 그다음 토큰 하나를 더 얻습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">그래도 확인하는 이유:</strong> 작은 층의 추측은 틀릴 수 있습니다. 틀린 초안을 그대로 쓰면 답이 목표 모델의 답과 달라지므로, 목표 모델이 확인한 초안만 답에 씁니다. 확인은 다시 만드는 일이 아니라, 목표 모델이 그 자리에 쓸 토큰과 초안을 맞대어 보는 일입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">수용 규칙:</strong> temperature=0(그리디 생성, 매번 확률이 가장 큰 토큰을 고르는 방식)에서는 목표 모델이 고른 토큰과 같은 초안만 수용합니다. 그래서 남는 토큰은 모두 목표 모델이 고른 토큰입니다. 다만 여러 토큰을 한 번에 검증하는 계산의 수치 차이로 실제 답은 기준 서버와 조금 달라질 수 있습니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/13_speculative_mtp.png" width="1050" alt="투기적 디코딩 예입니다. 1단계에서 MTP 층이 '근로'까지의 계산 결과로 초안 '시간은', '하루'를 예측하고, 2단계에서 목표 모델이 가중치를 한 번 읽어 두 초안이 맞는지 확인하며 '8시간을'을 더합니다. 단계 1번에 토큰 3개가 답에 붙습니다.">

<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인해도 빨라지는 이유</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">목표 모델은 초안 여러 개를 한 단계(가중치를 한 번 읽는 동안)에 함께 확인합니다. 한 단계의 시간은 대부분 가중치 읽기라, 토큰 하나를 만들 때와 시간이 거의 같습니다.</li>
<li style="margin:3px 0;">그림처럼 초안 2개가 모두 맞으면 단계 1번으로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">시간은 / 하루 / 8시간을</code> 3토큰을 얻습니다. 일반 decode는 같은 3토큰에 단계 3번이 필요합니다.</li>
<li style="margin:3px 0;">둘째 초안이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">하루가</code>로 틀렸다면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">시간은</code>만 받고 틀린 자리에는 목표 모델이 고른 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">하루</code>를 써, 단계 1번으로 2토큰을 얻습니다. 초안이 모두 틀려도 목표 모델이 고른 토큰 1개는 얻으므로, 잃는 것은 초안을 만든 시간뿐입니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">수용 길이(acceptance length)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">검증 한 번에 얻는 토큰 수의 평균으로, 1 + 수용된 초안 수의 평균입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">초안 수용률(draft acceptance rate):</strong> 만든 초안 토큰 가운데 수용된 초안 토큰의 비율입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 JSON의 키:</strong> 투기적 디코딩을 켠 서버를 벤치마크하면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code>가 서버의 초안 통계를 읽어, 결과 JSON에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">spec_decode_acceptance_length</code>(수용 길이)와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">spec_decode_acceptance_rate</code>(초안 수용률)를 저장합니다. 투기적 디코딩을 켜지 않은 서버의 결과에는 이 키가 없습니다.</li>
</ul>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">수용 길이</strong> = 1 + 수용된 초안 토큰 수 ÷ 검증 횟수</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">초안을 2개씩 만들어 검증을 네 번 했고, 수용된 초안이 차례로 2개, 1개, 0개, 2개인 경우를 계산해 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">수용 길이:</strong> 1 + (2 + 1 + 0 + 2) / 4 = 2.25토큰입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">초안 수용률:</strong> 만든 초안 8개 가운데 5개라 62.5%입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">순전파 수:</strong> 검증 네 번에 토큰 9개를 얻었으므로, 목표 모델이 혼자 토큰 9개를 만들 때(순전파 9번)보다 순전파가 다섯 번 적습니다.</li>
</ol>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">MTP 서버 설정</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 옵션:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--speculative-config</code>에 JSON 문자열로 넣습니다. 이 교안은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;method&quot;: &quot;mtp&quot;, &quot;num_speculative_tokens&quot;: 2}</code>(초안 2개)입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">대가:</strong> MTP 층도 GPU 메모리를 써, 같은 GPU에서 KV 캐시에 담는 토큰이 줄어듭니다(8-1 측정 예).</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실무에서 쓰는 때</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">많이 쓰이는 이유:</strong> DeepSeek-V3, Qwen3.5처럼 MTP 층을 함께 내놓는 모델이 늘어, 초안 모델 없이 옵션 하나로 켤 수 있는 투기적 디코딩으로 자주 쓰입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">잘 맞는 경우:</strong> 동시 요청이 적고 답이 길며, 한 사용자에게 답이 빨리 이어지는 것이 중요한 서비스입니다. 예: 대화형 상담 봇, 코딩 도우미.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">맞지 않는 경우:</strong> 요청이 몰려 GPU 계산이 이미 꽉 찬 대량 처리(문서 일괄 요약, 오프라인 평가)처럼 처리량이 중요한 작업입니다. KV 캐시 메모리가 빠듯할 때도 손해일 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">켜기 전에:</strong> 예상 동시 요청 수에서 기준 설정과 같은 부하로 측정해 목표 달성률과 출력 처리량을 비교합니다(8절).</li>
</ul>
<p style="margin:6px 0;">초안 모델 없이 쓸 수 있는 방법은 MTP와 n-gram 투기 두 가지입니다. 이 교안은 MTP를 측정합니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">방법</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">초안을 만드는 곳</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">잘 맞는 경우</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">MTP</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">모델 안의 MTP 층</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">그 모델에 MTP 층이 있을 때</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">n-gram 투기(n-gram speculation)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">답의 직전 토큰 몇 개(n-gram)와 같은 부분을 입력과 지금까지 만든 답에서 찾아, 그 뒤 토큰</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">답이 입력의 문장을 그대로 옮길 때(요약, 인용)</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">이 교안은 n-gram을 설명만 하고 측정하지 않습니다. 답이 입력의 문장을 그대로 옮기는 서비스라면 같은 sweep에 n-gram 설정을 더해 기준과 측정해 비교합니다.</p>
</div>

## 7. sweep 결과 읽기

### 7-1. 설정별 KV 캐시와 블록 크기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">3절에서 실행한 sweep이 끝나 터미널 1에 다시 명령을 입력할 수 있게 되면(맨 아래 줄에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">$</code> 표시가 보이면) 아래 명령을 실행합니다.</p>
</div>

**RunPod 터미널 1 · 화면에서 확인**

```bash
grep -E "GPU KV cache size|attention block size" output/sweep/labor-settings.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">줄 순서:</strong> 서버마다 attention 블록 크기 줄과 KV 캐시 토큰 수 줄이 이 순서로 한 번씩, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serve_params</code> 순서(기준, MTP)대로 나옵니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시 토큰 수:</strong> GPU 메모리를 같은 비율(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--gpu-memory-utilization</code> 기본값)만큼 써도 MTP 서버는 MTP 층이 메모리를 써 기준보다 적게 담습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">attention 블록 크기:</strong> 기준 서버의 블록 크기는 2-2 화면의 숫자와 같습니다. MTP 서버의 블록 크기가 기준과 다르면 접두어 캐싱이 다시 쓰는 토큰 수도 달라집니다.</li>
</ul>
</div>

### 7-2. summary.csv 읽기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">sweep이 저장한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/sweep/labor-settings/summary.csv</code>를 같은 원격 노트북에서 바로 읽습니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>에는 결과 JSON의 키(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_throughput</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">p95_ttft_ms</code> 등)가 열로, 서버 설정 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_benchmark_name</code>)과 벤치마크 조건(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dataset_path</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_concurrency</code>)이 함께 들어 있습니다. 첫 열은 이름 없는 행 번호라 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">index_col=0</code>으로 읽습니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code> 한 행에서 비교 테이블의 한 행을 만드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_row</code>를 함수로 둡니다. 8절과 9절 함께 따라하기에서 다시 씁니다. 목표 달성률(attainment)은 교안 01과 같이 goodput을 요청 처리량으로 나눈 값입니다. 요청 1개 완료는 질문 하나의 응답을 마지막 토큰까지 받은 것입니다(교안 01 2절).</p>
</div>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
def bench_row(setting, concurrency, result):
    """결과 JSON 딕셔너리나 summary.csv의 한 행에서 비교 테이블의 한 행을 만듭니다."""
    return {
        "설정": setting,
        "동시 요청 수": concurrency,
        "출력 처리량(토큰/초)": result["output_throughput"],
        "p95 TTFT(ms)": result["p95_ttft_ms"],
        "p95 TPOT(ms)": result["p95_tpot_ms"],
        "goodput(요청/초)": result["request_goodput"],
        "평균 TTFT(ms)": result["mean_ttft_ms"],
        "평균 TPOT(ms)": result["mean_tpot_ms"],
        "목표 달성률": result["request_goodput"] / result["request_throughput"],  # 두 지연 목표를 모두 지킨 요청의 비율
    }

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 입력 파일의 행만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_table</code>에 담고, 전월세 입력 파일의 행은 9절 함께 따라하기에서 씁니다.</p>
</div>

In [ ]:
# 실험 폴더의 summary.csv: 측정 하나가 한 행입니다(첫 열은 행 번호)
sweep_results = pd.read_csv(sweep_dir / "labor-settings" / "summary.csv", index_col=0)

# 노무 입력 파일의 행마다 비교 테이블의 한 행을 만듭니다. 수용 길이는 투기적 디코딩 서버의 행에만 값이 있고 나머지는 빈칸(NaN)입니다
sweep_rows = []
for result in sweep_results.to_dict("records"):
    if result["dataset_path"] == "output/labor_prompts.jsonl":
        row = bench_row(result["_benchmark_name"], result["max_concurrency"], result)
        row["수용 길이"] = result["spec_decode_acceptance_length"]
        sweep_rows.append(row)

sweep_table = pd.DataFrame(sweep_rows).round(2)
display(sweep_table)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">노무 4행:</strong> 설정 2개 x 동시 요청 수 2개(1, 64)가 모두 있습니다. 설정 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serve_params</code>의 키 그대로입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">수용 길이:</strong> MTP 행에만 값이 있습니다.</li>
</ul>
</div>

## 8. 기준과 MTP 비교, 서버 설정 정하기

### 8-1. 투기적 디코딩: 낮은 부하와 높은 부하의 기준과 MTP

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">7-2의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_table</code>에서 같은 동시 요청 수끼리 기준 행과 MTP 행을 비교합니다. 수용 길이는 MTP 행에만 있습니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 예: 기준 서버와 MTP 서버</strong></p>
<p style="margin:6px 0;">RunPod L4 24GB, vLLM 0.27.0에서 3-2와 같은 조건(노무 입력, 동시 요청 1은 요청 8개, 동시 요청 64는 요청 64개)으로 기준 서버와 MTP 서버를 sweep으로 측정한 예입니다. 직접 다시 측정하지 않으며, GPU가 다르면 숫자도 달라집니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">동시 요청 수</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">출력 처리량(토큰/초)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">평균 TTFT(ms)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">평균 TPOT(ms)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">목표 달성률</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">수용 길이</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">기준</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">27.3</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">213</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">35.6</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1.000</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">-</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">MTP</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">39.9</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">392</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">22.5</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">0.875</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">2.32</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">기준</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">64</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">485.3</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">2,497</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">82.6</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">0.000</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">-</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">MTP</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">64</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">368.2</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">7,559</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">77.7</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">0.000</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">2.30</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결론: 이 측정 예에서는 두 부하 모두 기준 설정이 낫습니다.</strong> MTP는 낮은 부하에서 답이 이어지는 속도(TPOT)를 높였지만 첫 토큰이 늦어져 지연 목표를 놓쳤고, 높은 부하에서는 출력 처리량까지 줄었습니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">동시 요청 수</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">MTP가 나은 점</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">MTP가 못한 점</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">고를 설정</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">TPOT 35.6에서 22.5ms, 출력 처리량 27.3에서 39.9토큰/초</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">요청 8개 중 1개의 TTFT가 목표 1초를 넘어(p99 TTFT 1,564ms, 기준은 916ms) 목표 달성률 1.000에서 0.875</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">기준(MTP는 0.95 미만)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">64</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">TPOT 82.6에서 77.7ms(차이 작음)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">출력 처리량 485.3에서 368.2토큰/초, 평균 TTFT 2,497에서 7,559ms</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">없음(둘 다 목표 달성률 0)</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">낮은 부하에서 빨라진 이유:</strong> 남는 계산 능력으로 초안을 확인해, 검증 한 번에 평균 2.32토큰(수용 길이)을 얻었습니다. 초안을 만들고 확인하는 계산도 들기 때문에 2.32배가 아니라 약 1.6배(TPOT 기준) 빨라졌습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">높은 부하에서 손해인 이유:</strong> GPU 계산이 이미 요청들로 차 있어 확인 계산이 그대로 부담이 됩니다. 또 MTP 층이 쓰는 메모리로 KV 캐시가 247,994토큰에서 146,870토큰으로 줄어, 자리를 기다리는 요청이 늘었습니다. sweep 로그의 한 줄에서 MTP 서버는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU KV cache usage</code>(KV 캐시 가운데 쓰고 있는 비율) 98.2%, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Waiting</code>(차례를 기다리는 요청 수) 18개였고, 기준 서버는 37.6%, 6개였습니다(부록 01 1절).</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실무 판단:</strong> MTP는 동시 요청이 적고 긴 답을 빨리 이어 보여 주는 것이 중요한 서비스에서 후보가 됩니다. 그때도 TTFT 목표와 함께 측정해, 목표 달성률로 고릅니다(8-2).</li>
</ul>
</div>

### 8-2. 서버 설정 정하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서비스의 예상 동시 요청 수에서 두 설정 가운데 하나를 고릅니다. 기준은 교안 01 5-3에서 동시 요청 수를 고를 때와 같고, 아래 코드는 이 기준을 그대로 옮긴 것입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">고르는 기준</strong></p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">같은 동시 요청 수끼리만 비교합니다:</strong> 예상 동시 요청 수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">expected_concurrency</code>)의 행만 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">지연 목표를 지킨 설정만 후보로 남깁니다:</strong> 목표 달성률(TTFT 1,000ms와 TPOT 50ms를 모두 지킨 요청의 비율)이 <strong style="font-size:1.13em;color:#1f3a5f;">0.95 이상</strong>인 설정만 후보입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">후보 가운데 출력 처리량이 가장 큰 설정을 고릅니다:</strong> 같은 GPU로 토큰을 더 많이 만들수록 출력 토큰당 비용이 싸기 때문입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">후보가 없으면 고르지 않습니다:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_setting</code>이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>으로 남고 안내 문장이 나옵니다.</li>
</ol>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">기준</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">코드의 조건</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1. 같은 동시 요청 수</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">row[&quot;동시 요청 수&quot;] == expected_concurrency</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">2. 목표 달성률 0.95 이상</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">row[&quot;목표 달성률&quot;] &gt;= 0.95</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">3. 출력 처리량이 가장 큰 설정</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">row[&quot;출력 처리량(토큰/초)&quot;] &gt; best_throughput</code>이면 그 설정으로 바꿔 둡니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">4. 후보가 없을 때</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">반복이 끝나도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_setting is None</code></td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">비교는 반올림 전 값인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_rows</code>로 합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_table</code>은 둘째 자리에서 반올림한 값이라 0.95 경계에서 결과가 달라질 수 있기 때문입니다.</p>
</div>

In [ ]:
expected_concurrency = 1  # 서비스의 예상 동시 요청 수(sweep의 노무 조건 1, 64 가운데 하나). 64로 바꿔 다시 실행하면 어떤 설정이 남는지(하나도 남지 않을 수도 있습니다) 비교할 수 있습니다

# 반올림 전 값인 sweep_rows에서 예상 동시 요청 수의 행만 보고 고릅니다(테이블은 둘째 자리에서 반올림한 값이라 반올림 전 값으로 비교합니다)
best_setting = None
best_throughput = 0
for row in sweep_rows:
    if row["동시 요청 수"] == expected_concurrency and row["목표 달성률"] >= 0.95 and row["출력 처리량(토큰/초)"] > best_throughput:
        best_setting = row["설정"]
        best_throughput = row["출력 처리량(토큰/초)"]

# 결정 근거 테이블: 예상 동시 요청 수의 두 설정
decision_table = sweep_table[sweep_table["동시 요청 수"] == expected_concurrency]
display(decision_table[["설정", "출력 처리량(토큰/초)", "p95 TTFT(ms)", "p95 TPOT(ms)", "목표 달성률"]])

# 고른 설정이 없으면 이 동시 요청 수에서는 어떤 설정도 지연 목표를 95% 이상 지키지 못한 것입니다
if best_setting is None:
    print(f"동시 요청 {expected_concurrency}에서 지연 목표를 95% 이상 지킨 설정이 없습니다. 지연 목표를 다시 정하거나, 서버 한 대에 보내는 동시 요청 수를 줄이거나 서버를 늘립니다.")
else:
    print(f"동시 요청 {expected_concurrency}에서 고른 설정: {best_setting} | 더할 옵션: {serve_params[best_setting]} | 출력 처리량: {best_throughput:.1f} 토큰/초")

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">best_setting</code>:</strong> 위 기준 1~4로 고른 설정입니다. 8-1의 측정 예라면 동시 요청 1에서는 기준(MTP는 목표 달성률 0.875라 후보에서 빠짐), 64에서는 후보가 없어 안내 문장이 나옵니다. 예상 동시 요청 수를 64로 바꿔 다시 실행해 비교합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 요청 1:</strong> 요청이 8개라 한 요청만 지연 목표를 넘어도 목표 달성률이 0.875(8개 중 7개)가 되어 탈락합니다. sweep은 측정마다 접두어 캐시를 비우므로 측정의 첫 요청은 시스템 메시지 전체를 prefill 해 TTFT가 길어집니다. 그래서 MTP의 TPOT가 더 짧아도 이런 요청 하나 때문에 기준이 골라지거나, 두 설정 모두 탈락할 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실무:</strong> 고른 설정의 옵션을 서버 명령 뒤에 붙이고, 결정 근거 테이블(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">decision_table</code>)과 함께 남깁니다. 같은 설정도 실행마다 값이 달라질 수 있으므로, 차이가 작으면 한 번의 측정만으로 우열을 단정하지 않습니다.</li>
</ul>
</div>

## 9. 🖐️ 함께 따라하기: 전월세 창구의 서버 설정 정하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">전월세 상담 창구의 예상 동시 요청 수에서 지연 목표를 지키는 서버 설정을, 같은 sweep 결과에서 기준과 MTP 가운데 고릅니다. 서버를 새로 띄우지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">데이터:</strong> 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>의 전월세 조건 행입니다. 입력 파일은 교안 01 함께 따라하기에서 생성한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/lease_prompts.jsonl</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">예상 동시 요청 수:</strong> 16입니다(전월세 조건은 동시 요청 16, 요청 16개).</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> 작성 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>만 주석에 맞춰 채웁니다. 본실습의 변수와 함수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_results</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serve_params</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_row</code>)를 그대로 쓰므로 본실습을 먼저 실행합니다. 완성 코드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/교안_02_정답.ipynb</code>의 같은 절에 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정:</strong> 서버 설정 두 개(기준, MTP)와 벤치마크 옵션(생성 상한 256토큰, 지연 목표 TTFT 1000ms, TPOT 50ms)은 본실습과 같습니다.</p>
</div>

### 9-1. 전월세 행 고르기

<details><summary>힌트</summary>

```text
접근방법: 7-2를 참고합니다.
summary.csv의 dataset_path 열 값이 3-2의 bench_params에 적은 전월세 입력 파일 경로와 같은 행만 고릅니다.
```

</details>

In [ ]:
# 함께 따라하기: dataset_path가 전월세 입력 파일인 행만 고르는 조건을 채우세요
lease_sweep_rows = []
for result in sweep_results.to_dict("records"):
    if ...:
        row = bench_row(result["_benchmark_name"], result["max_concurrency"], result)
        row["수용 길이"] = result["spec_decode_acceptance_length"]
        lease_sweep_rows.append(row)

lease_sweep_table = pd.DataFrame(lease_sweep_rows).round(2)
display(lease_sweep_table)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_sweep_table</code>이 두 행(기준, MTP)이고, 동시 요청 수가 모두 16입니다.</p>
</div>

### 9-2. 전월세 창구의 설정 고르기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">고르는 기준:</strong> 8-2와 같습니다. 목표 달성률이 0.95 이상인 설정만 후보로 남기고, 그 가운데 출력 처리량이 가장 큰 설정을 고릅니다. 후보가 없으면 고르지 않습니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>).</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 8-2를 참고합니다.
전월세 행은 모두 동시 요청 16이므로 목표 달성률과 지금까지의 최대 출력 처리량, 두 조건만 비교합니다.
```

</details>

In [ ]:
# 함께 따라하기: 8-2와 같은 규칙(목표 달성률 0.95 이상, 출력 처리량 최대)의 두 조건을 채우세요
lease_best_setting = None
lease_best_throughput = 0
for row in lease_sweep_rows:
    if ... and ...:
        lease_best_setting = row["설정"]
        lease_best_throughput = row["출력 처리량(토큰/초)"]

# 고른 설정이 없으면 동시 요청 16에서 어떤 설정도 지연 목표를 95% 이상 지키지 못한 것입니다
if lease_best_setting is None:
    print("동시 요청 16에서 지연 목표를 95% 이상 지킨 설정이 없습니다. 서버 한 대에 보내는 동시 요청 수를 줄이거나 서버를 늘립니다.")
else:
    print(f"전월세 창구의 설정: {lease_best_setting} | 더할 옵션: {serve_params[lease_best_setting]} | 출력 처리량: {lease_best_throughput:.1f} 토큰/초")

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">출력 처리량이 더 큰 설정도 목표 달성률이 0.95 미만이면 고르지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: 고른 설정은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_sweep_table</code>에서 목표 달성률이 0.95 이상인 행 가운데 출력 처리량이 가장 큰 행의 설정과 같습니다. 0.95 이상인 행이 없을 때만 안내 문장이 나옵니다.</p>
<p style="margin:6px 0;">sweep은 서버 설정마다 측정을 마치면 그 서버를 끄므로, 마지막 MTP 서버도 sweep이 끝날 때 꺼져 있습니다. RunPod 터미널 1에서 GPU 메모리가 반환됐는지 확인합니다.</p>
</div>

**RunPod 터미널 1 · 화면에서 확인**

```bash
nvidia-smi
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: GPU 메모리 사용량이 2-3에서 상담 서버를 끈 뒤와 같은 수준입니다. 메모리가 남아 있으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pgrep -af &quot;vllm serve&quot;</code>로 남은 vLLM 프로세스의 PID를 찾아 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">kill &lt;PID&gt;</code>로 끕니다.</p>
</div>

## 10. 핵심 정리

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">접두어 캐싱(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>):</strong> vLLM 0.27.0은 Qwen3.5 같은 하이브리드 모델에서 기본으로 꺼 두므로 직접 켭니다. 공통 부분을 앞에, 요청마다 다른 부분을 뒤에 두면 앞에서부터 같은 KV 캐시 블록을 다시 써 prefill을 줄입니다. 측정 예(L4, 동시 요청 16)에서는 캐싱을 켜면 같은 요청 64개를 약 2.5배 빨리 끝냈습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시 자료형(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--kv-cache-dtype fp8</code>):</strong> KV 캐시 값을 1바이트로 저장해 같은 메모리에 담는 토큰을 늘립니다. Qwen3.5는 attention 층의 KV만 줄어 2배보다 적게 늘고, attention 블록이 커져 캐시 적중이 줄면 TTFT가 늘 수 있습니다. 이 교안은 측정하지 않았고, 동시 요청이 많아 KV 캐시가 모자랄 때 측정해 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 실행 상한(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--max-num-seqs</code>):</strong> 함께 계산하는 요청 수를 제한해 TPOT를 줄일 수 있지만, 대기 시간이 더해져 TTFT가 길어지거나 출력 처리량이 줄 수 있습니다. 실제 요청 수가 상한에 못 미치면 효과가 없습니다. 이 교안에서는 원리만 설명하며, 요청이 몰릴 때 같은 부하로 측정해 비교합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">투기적 디코딩(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--speculative-config</code>):</strong> 초안 토큰을 목표 모델이 한 번에 검증해, 검증 한 번에 평균 수용 길이만큼 토큰을 얻습니다. 초안을 만들고 검증하는 계산이 대가라, 수용 길이가 1에 가깝거나 동시 요청이 많으면 출력 처리량에서 손해일 수 있습니다. 8-1의 측정 예에서는 두 부하 모두 TTFT가 길어져, 낮은 부하에서 TPOT와 출력 처리량이 앞서도 지연 목표로 고르면 기준에 질 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code>:</strong> 서버 설정마다 서버를 한 번 띄우고, 벤치마크 조건을 차례로 측정한 뒤 끕니다. 이 과정을 명령 하나로 반복해 결과를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>에 모읍니다. 설정 2개와 조건 3개이면 서버 시작은 2번, 측정은 6번입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 설정 정하기:</strong> 예상 동시 요청 수에서 기준과 MTP를 비교해, 목표 달성률 0.95 이상인 설정 가운데 출력 처리량이 가장 큰 설정을 고르고, 그 옵션과 결정 근거 테이블을 함께 남깁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 위치:</strong> 같은 VS Code 원격 창에서 서버·sweep·로그 확인은 터미널로, 요청·설정 파일 생성·<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code> 분석은 노트북으로 진행합니다. 입력과 결과 파일은 모두 같은 파드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/</code>에 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 시작 줄이기:</strong> 서버 시작은 느리므로 같은 설정의 서버는 이어 쓰고, 설정을 바꿀 때만 sweep으로 몰아서 띄웁니다.</li>
</ul>
</div>